# NDVI CDR Pyramid — Migrate v1 → Icechunk v2

Copies the existing v1 GeoZarr pyramid store (`ndvi-cdr-pyramid`) to a new prefix
(`ndvi-cdr-pyramid-v2`) using boto3, then upgrades the copy in-place to Icechunk v2
format using `ic.upgrade_icechunk_repository()`. The original v1 store is untouched.

| | |
|---|---|
| **Source (v1)** | `s3://osc-pub/ndvi-cdr-pyramid` |
| **Destination (v2)** | `s3://osc-pub/ndvi-cdr-pyramid-v2` |
| **Public URL** | `https://r2-pub.openscicomp.io/ndvi-cdr-pyramid-v2` |

## 1. Setup

In [ ]:
import os
from dotenv import load_dotenv
import boto3
import icechunk as ic

load_dotenv(f'{os.environ["HOME"]}/dotenv/r2-osc-pub.env')

r2_bucket   = "osc-pub"
r2_endpoint = os.environ["ENDPOINT_URL"]
src_prefix  = "ndvi-cdr-pyramid"
dst_prefix  = "ndvi-cdr-pyramid-v2"

print(f"Source : s3://{r2_bucket}/{src_prefix}")
print(f"Dest   : s3://{r2_bucket}/{dst_prefix}")

## 2. Copy the v1 store to a new prefix

Uses the S3 server-side copy API (no data downloads) to duplicate every object
under `ndvi-cdr-pyramid/` to `ndvi-cdr-pyramid-v2/`.

In [ ]:
s3 = boto3.client(
    "s3",
    endpoint_url=r2_endpoint,
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    region_name="auto",
)

paginator = s3.get_paginator("list_objects_v2")
pages = paginator.paginate(Bucket=r2_bucket, Prefix=src_prefix + "/")

copied = 0
for page in pages:
    for obj in page.get("Contents", []):
        src_key = obj["Key"]
        dst_key = dst_prefix + src_key[len(src_prefix):]
        s3.copy_object(
            Bucket=r2_bucket,
            CopySource={"Bucket": r2_bucket, "Key": src_key},
            Key=dst_key,
        )
        copied += 1
        if copied % 500 == 0:
            print(f"  copied {copied} objects…")

print(f"Done — copied {copied} objects to s3://{r2_bucket}/{dst_prefix}")

## 3. Dry-run migration (no writes)

Validates the migration process without committing any changes.

In [ ]:
dst_storage = ic.s3_storage(
    bucket=r2_bucket,
    prefix=dst_prefix,
    from_env=True,
    endpoint_url=r2_endpoint,
    region="auto",
    force_path_style=False,
)

dst_repo = ic.Repository.open(dst_storage)
print(f"Opened copy — spec version: {dst_repo.spec_version}")

# Dry run: validate without writing
migrated = ic.upgrade_icechunk_repository(dst_repo, dry_run=True)
print("Dry run passed — ready to migrate")

## 4. Migrate to Icechunk v2 (irreversible on this copy)

`dry_run=False` writes the v2-format files.  The original `ndvi-cdr-pyramid` is unchanged.

In [ ]:
dst_repo2   = ic.Repository.open(dst_storage)
migrated    = ic.upgrade_icechunk_repository(dst_repo2, dry_run=False)

assert migrated.spec_version == 2, f"Expected v2, got {migrated.spec_version}"
print(f"Migration complete — spec version: {migrated.spec_version}")

## 5. Verify — read back and inspect

In [ ]:
import zarr
import xarray as xr

session = migrated.readonly_session("main")
root = zarr.open_group(session.store, mode="r")
print(root.tree())
print("\nmultiscales metadata:")
print(root.attrs.get("multiscales", root.attrs))

In [ ]:
import hvplot.xarray  # noqa

ds_coarse = xr.open_zarr(session.store, group="3", consolidated=False)
ds_coarse["NDVI"].isel(time=0).hvplot(
    rasterize=True, geo=True, global_extent=True,
    x="longitude", y="latitude", tiles="OSM",
    cmap="YlGn", clim=(-0.1, 1.0),
    title="AVHRR NDVI — 2000-01-01 (level 3, v2 store)",
    width=800, height=400,
)

## 6. Explorer URL

Test the v2 store in the icechunk-explorer:

In [ ]:
base = "https://opensciencecomputing.github.io/NCICS-2026/icechunk-explorer/"
store_url = f"https://r2-pub.openscicomp.io/{dst_prefix}"
explorer_url = f"{base}?url={store_url}&var=NDVI"
print(explorer_url)